# Parsl Monte Carlo Pi

This workflow adapts the **Monte Carlo workflow** from the official [`1-parsl-introduction.ipynb`](https://github.com/Parsl/parsl-tutorial/blob/71fbd34d826bf60174fbab3a5213e4e9ed80640f/1-parsl-introduction.ipynb). Three independent Parsl Apps estimate pi, and a dependent App computes their mean. Floability supplies the portable environment and TaskVine workers.

## Configure Parsl's TaskVine executor

Floability starts `vine_factory` before this notebook runs. Therefore Parsl uses `worker_launch_method="manual"`: Parsl creates the TaskVine manager and executes the Apps, while Floability remains responsible for worker submission. Both sides meet through the manager name in `VINE_MANAGER_NAME`.

In [1]:
import atexit
import json
import math
import os
import socket
import time
import uuid
from datetime import datetime
from pathlib import Path
from zoneinfo import ZoneInfo

import parsl
from parsl import python_app
from parsl.config import Config
from parsl.executors.taskvine import TaskVineExecutor, TaskVineManagerConfig

print(f"Parsl version: {parsl.__version__}")

Parsl version: 2026.08.10


In [2]:
POINTS_PER_TASK = 1_000_000
INTRODUCTORY_SEEDS = (202601, 202602, 202603)

# Increase this to compare a larger independent ensemble (maximum 1,000).
SCALED_TASKS = 20
MAX_SCALED_TASKS = 1_000
SCALED_SEED_START = 202604
MAX_OUTPUT = 20

if POINTS_PER_TASK <= 0:
    raise ValueError("POINTS_PER_TASK must be positive")
if not 3 <= SCALED_TASKS <= MAX_SCALED_TASKS:
    raise ValueError(
        f"SCALED_TASKS must be between 3 and {MAX_SCALED_TASKS:,}"
    )
if MAX_OUTPUT < 0:
    raise ValueError("MAX_OUTPUT cannot be negative")

print(f"Points per task: {POINTS_PER_TASK:,}")
print(f"Introductory tasks: {len(INTRODUCTORY_SEEDS)}")
print(f"Scaled tasks: {SCALED_TASKS}")
print(f"Maximum printed worker assignments per stage: {MAX_OUTPUT}")

Points per task: 1,000,000
Introductory tasks: 3
Scaled tasks: 20
Maximum printed worker assignments per stage: 20


In [3]:
def choose_manager_port(port_spec):
    """Choose one available TCP port from Floability's inclusive range."""
    values = [int(value.strip()) for value in port_spec.split(",") if value.strip()]
    if not values:
        raise ValueError("VINE_MANAGER_PORTS does not contain a port")

    start = values[0]
    end = values[-1]
    if start > end:
        start, end = end, start

    for port in range(start, end + 1):
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as candidate:
            candidate.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
            try:
                candidate.bind(("", port))
            except OSError:
                continue
            return port

    raise RuntimeError(f"No available manager port in {start}:{end}")


manager_name = os.environ.get("VINE_MANAGER_NAME")
if not manager_name:
    raise RuntimeError("VINE_MANAGER_NAME is not set; run this notebook through Floability")

manager_port_spec = os.environ.get("VINE_MANAGER_PORTS", "9123,9150")
manager_port = choose_manager_port(manager_port_spec)
print(f"TaskVine manager name: {manager_name}")
print(f"TaskVine manager port: {manager_port}")

TaskVine manager name: floability-7ba42f0e-dde6-45ad-b2bc-dcacbffccfcd
TaskVine manager port: 9123


In [4]:
OUTPUT_DIR = Path("outputs")
PARSL_RUN_DIR = OUTPUT_DIR / "parsl-runinfo"
TASKVINE_LOG_DIR = PARSL_RUN_DIR / "taskvine-manager"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

try:
    parsl.clear()
except Exception:
    pass

taskvine_executor = TaskVineExecutor(
    label="floability-taskvine",
    worker_launch_method="manual",
    function_exec_mode="regular",
    manager_config=TaskVineManagerConfig(
        project_name=manager_name,
        port=manager_port,
        max_retries=1,
        wait_for_workers=1,
        vine_log_dir=str(TASKVINE_LOG_DIR),
    ),
)

config = Config(
    executors=[taskvine_executor],
    retries=1,
    run_dir=str(PARSL_RUN_DIR),
    usage_tracking=0,
)
parsl.load(config)
atexit.register(parsl.clear)
print("Parsl loaded with the TaskVine executor in manual worker mode.")

Parsl loaded with the TaskVine executor in manual worker mode.


## Estimate pi in parallel

The following Apps retain the official tutorial's calculation and dependency shape. For Floability adoption, the `pi()` App adds a seed argument for repeatable validation. The surrounding cells add the [official Parsl TaskVine executor](https://github.com/Parsl/parsl/blob/b8b474c7a6ee6fd2f9392e10291c2d8c323b0510/parsl/configs/vineex_local.py) in manual-worker mode, environment-based manager discovery, validation, timing, and JSON output. Each `pi()` invocation remains independent; `mean()` cannot run until all three estimates are available.

In [5]:
# Adapted from the Apache-2.0 Parsl tutorial Monte Carlo example.
@python_app
def pi(num_points, seed):
    from random import Random

    random = Random(seed).random
    inside = 0
    for _ in range(num_points):
        x, y = random(), random()  # Drop a random point in the box.
        if x**2 + y**2 < 1:        # Count points within the circle.
            inside += 1

    return inside * 4 / num_points


@python_app
def mean(a, b, c):
    return (a + b + c) / 3


print("=" * 60)
print("Starting the tutorial-style three-task computation...")
print("=" * 60)
intro_started_at = time.perf_counter()

a, b, c = (pi(POINTS_PER_TASK, seed) for seed in INTRODUCTORY_SEEDS)
mean_pi = mean(a, b, c)

Starting the tutorial-style three-task computation...


## Collect and validate the result

Calling `.result()` at the final boundary propagates any Parsl or TaskVine task failure. The notebook then checks the numerical result before reporting success. Parsl futures do not expose TaskVine's `done.addrport` property directly, so the following helper reads the manager's TaskVine transaction log to report the same manager-observed worker connection endpoints.

In [6]:
intro_estimates = [a.result(), b.result(), c.result()]
intro_average = mean_pi.result()
intro_execution_time = time.perf_counter() - intro_started_at
intro_absolute_error = abs(intro_average - math.pi)

if not all(math.isfinite(value) for value in [*intro_estimates, intro_average]):
    raise RuntimeError("Monte Carlo workflow returned a non-finite result")
if intro_absolute_error > 0.02:
    raise RuntimeError(
        f"Monte Carlo estimate {intro_average:.6f} is unexpectedly far from pi "
        f"(absolute error {intro_absolute_error:.6f})"
    )

print(f"a: {intro_estimates[0]:.5f} b: {intro_estimates[1]:.5f} c: {intro_estimates[2]:.5f}")
print(f"Average of three runs: {intro_average:.5f}")
print(f"math.pi: {math.pi:.5f}")
print(f"Absolute error: {intro_absolute_error:.6f}")
print(f"Execution time: {intro_execution_time:.2f} seconds")

a: 3.14440 b: 3.14078 c: 3.14442
Average of three runs: 3.14320
math.pi: 3.14159
Absolute error: 0.001605
Execution time: 7.12 seconds


In [7]:
def read_taskvine_worker_assignments():
    """Read task-to-worker endpoints from TaskVine's transaction log."""
    transaction_logs = list(TASKVINE_LOG_DIR.rglob("transactions"))
    if not transaction_logs:
        raise RuntimeError(
            f"TaskVine transaction log not found under {TASKVINE_LOG_DIR}"
        )

    transaction_log = max(
        transaction_logs,
        key=lambda path: path.stat().st_mtime_ns,
    )
    worker_endpoints = {}
    task_categories = {}
    task_workers = {}

    for line in transaction_log.read_text().splitlines():
        fields = line.split()
        if len(fields) < 6 or fields[0].startswith("#"):
            continue
        if fields[2] == "WORKER" and fields[4] == "CONNECTION":
            worker_endpoints[fields[3]] = fields[5]
        elif fields[2] == "TASK" and fields[4] == "READY":
            task_categories[int(fields[3])] = fields[5]
        elif fields[2] == "TASK" and fields[4] == "RUNNING":
            task_workers[int(fields[3])] = fields[5]

    return [
        {
            "task_id": task_id,
            "category": task_categories.get(task_id, "unknown"),
            "worker": worker_endpoints.get(worker_id, worker_id),
        }
        for task_id, worker_id in sorted(task_workers.items())
    ]


def add_pi_estimates(assignments, estimates):
    """Attach pi results to the corresponding submitted pi tasks."""
    pi_estimates = iter(estimates)
    enriched_assignments = []
    for assignment in assignments:
        enriched_assignment = assignment.copy()
        if assignment["category"] == "pi":
            try:
                enriched_assignment["pi_estimate"] = next(pi_estimates)
            except StopIteration as error:
                raise RuntimeError("Missing pi result for TaskVine assignment") from error
        enriched_assignments.append(enriched_assignment)

    try:
        next(pi_estimates)
    except StopIteration:
        return enriched_assignments
    raise RuntimeError("Missing TaskVine assignment for a pi result")


def print_worker_assignments(title, assignments):
    print(title)
    for assignment in assignments[:MAX_OUTPUT]:
        pi_output = (
            f" pi={assignment['pi_estimate']:.6f}"
            if "pi_estimate" in assignment
            else ""
        )
        print(
            f"  TaskVine task {assignment['task_id']:>3} "
            f"({assignment['category']}):{pi_output} "
            f"worker={assignment['worker']}"
        )
    omitted = len(assignments) - MAX_OUTPUT
    if omitted > 0:
        print(f"  ... {omitted} additional assignment(s) omitted")


intro_worker_assignments = add_pi_estimates(
    read_taskvine_worker_assignments(),
    intro_estimates,
)
intro_task_ids = {item["task_id"] for item in intro_worker_assignments}
print_worker_assignments(
    "Introductory TaskVine worker assignments:",
    intro_worker_assignments,
)

Introductory TaskVine worker assignments:
  TaskVine task   1 (pi): pi=3.144396 worker=172.17.0.2:55242
  TaskVine task   2 (pi): pi=3.140776 worker=172.17.0.2:55256
  TaskVine task   3 (pi): pi=3.144420 worker=172.17.0.2:55260
  TaskVine task   4 (mean): worker=172.17.0.2:55242


## Scale the experiment

Now launch a separate ensemble of `SCALED_TASKS` independent `pi()` Apps and reduce all of their results to one average. The default is 20 tasks, configurable from 3 through 1,000 in the cell near the top. Monte Carlo uncertainty decreases approximately as $1 / \sqrt{N}$ with the total number of sampled points. More workers reduce elapsed time; they do not change the statistical accuracy.

In [8]:
@python_app
def mean_many(inputs=()):
    return sum(inputs) / len(inputs)


print("=" * 60)
print(f"Starting the scaled computation with {SCALED_TASKS} tasks...")
print("=" * 60)
scaled_started_at = time.perf_counter()
scaled_futures = [
    pi(POINTS_PER_TASK, SCALED_SEED_START + task_index)
    for task_index in range(SCALED_TASKS)
]
scaled_mean_future = mean_many(inputs=scaled_futures)

Starting the scaled computation with 20 tasks...


In [9]:
scaled_estimates = [future.result() for future in scaled_futures]
scaled_average = scaled_mean_future.result()
scaled_execution_time = time.perf_counter() - scaled_started_at
scaled_absolute_error = abs(scaled_average - math.pi)
scaled_total_points = SCALED_TASKS * POINTS_PER_TASK

if not all(math.isfinite(value) for value in [*scaled_estimates, scaled_average]):
    raise RuntimeError("Scaled Monte Carlo workflow returned a non-finite result")
if scaled_absolute_error > 0.02:
    raise RuntimeError(
        f"Scaled estimate {scaled_average:.6f} is unexpectedly far from pi "
        f"(absolute error {scaled_absolute_error:.6f})"
    )

estimated_inside_probability = scaled_average / 4
scaled_standard_error = 4 * math.sqrt(
    estimated_inside_probability
    * (1 - estimated_inside_probability)
    / scaled_total_points
)
scaled_ci95 = (
    scaled_average - 1.96 * scaled_standard_error,
    scaled_average + 1.96 * scaled_standard_error,
)

print(f"Scaled average ({SCALED_TASKS} runs): {scaled_average:.6f}")
print(f"Total sampled points: {scaled_total_points:,}")
print(f"Absolute error: {scaled_absolute_error:.6f}")
print(
    f"Estimated 95% sampling interval: "
    f"[{scaled_ci95[0]:.6f}, {scaled_ci95[1]:.6f}]"
)
print(f"Execution time: {scaled_execution_time:.2f} seconds")
print(
    f"Error comparison: {intro_absolute_error:.6f} (three runs) -> "
    f"{scaled_absolute_error:.6f} ({SCALED_TASKS} runs)"
)

current_worker_assignments = read_taskvine_worker_assignments()
scaled_worker_assignments = add_pi_estimates([
    item
    for item in current_worker_assignments
    if item["task_id"] not in intro_task_ids
], scaled_estimates)
all_worker_assignments = intro_worker_assignments + scaled_worker_assignments
print_worker_assignments(
    "Scaled TaskVine worker assignments:",
    scaled_worker_assignments,
)

Scaled average (20 runs): 3.142237
Total sampled points: 20,000,000
Absolute error: 0.000645
Estimated 95% sampling interval: [3.141518, 3.142957]
Execution time: 2.77 seconds
Error comparison: 0.001605 (three runs) -> 0.000645 (20 runs)
Scaled TaskVine worker assignments:
  TaskVine task   5 (pi): pi=3.143072 worker=172.17.0.2:55242
  TaskVine task   6 (pi): pi=3.142172 worker=172.17.0.2:55256
  TaskVine task   7 (pi): pi=3.141768 worker=172.17.0.2:55242
  TaskVine task   8 (pi): pi=3.139872 worker=172.17.0.2:55256
  TaskVine task   9 (pi): pi=3.144420 worker=172.17.0.2:55260
  TaskVine task  10 (pi): pi=3.140916 worker=172.17.0.2:55262
  TaskVine task  11 (pi): pi=3.142160 worker=172.17.0.2:55256
  TaskVine task  12 (pi): pi=3.144536 worker=172.17.0.2:55260
  TaskVine task  13 (pi): pi=3.143160 worker=172.17.0.2:55242
  TaskVine task  14 (pi): pi=3.142168 worker=172.17.0.2:55262
  TaskVine task  15 (pi): pi=3.142872 worker=172.17.0.2:55256
  TaskVine task  16 (pi): pi=3.140676 worker

In [10]:
finished_at = datetime.now(ZoneInfo("America/New_York"))
summary = {
    "manager_name": manager_name,
    "manager_port": manager_port,
    "parsl_version": parsl.__version__,
    "configuration": {
        "points_per_task": POINTS_PER_TASK,
        "scaled_task_limit": MAX_SCALED_TASKS,
        "max_printed_worker_assignments_per_stage": MAX_OUTPUT,
    },
    "introductory": {
        "task_count": len(INTRODUCTORY_SEEDS),
        "total_points": len(INTRODUCTORY_SEEDS) * POINTS_PER_TASK,
        "seeds": list(INTRODUCTORY_SEEDS),
        "estimates": intro_estimates,
        "average": intro_average,
        "absolute_error": intro_absolute_error,
        "execution_time_seconds": intro_execution_time,
    },
    "scaled": {
        "task_count": SCALED_TASKS,
        "total_points": scaled_total_points,
        "seed_start": SCALED_SEED_START,
        "estimates": scaled_estimates,
        "average": scaled_average,
        "absolute_error": scaled_absolute_error,
        "estimated_standard_error": scaled_standard_error,
        "estimated_95_percent_interval": list(scaled_ci95),
        "execution_time_seconds": scaled_execution_time,
    },
    "taskvine_worker_assignments": all_worker_assignments,
    "finished_at": finished_at.isoformat(),
}
summary_path = OUTPUT_DIR / "monte-carlo-summary.json"
summary_path.write_text(json.dumps(summary, indent=2) + "\n")

parsl.clear()

execution_id = uuid.uuid4().hex[:8]
display_time = finished_at.strftime("%Y-%m-%d %I:%M:%S %p")
print("=" * 60)
print("COMPUTATION COMPLETE!")
print(
    f"Distributed execution time: "
    f"{intro_execution_time + scaled_execution_time:.2f} seconds"
)
print(f"Output saved to: {summary_path}")
print(f"__floability_execution_done__::{display_time}::{execution_id}")
print("=" * 60)

COMPUTATION COMPLETE!
Distributed execution time: 9.89 seconds
Output saved to: outputs/monte-carlo-summary.json
__floability_execution_done__::2026-08-19 06:49:00 PM::d27c39d5
